# Lecture 5.4: UserPromptSubmit, Injecting Context

Every hook you have written so far reacted to something the agent did, such as running a tool. The `UserPromptSubmit` hook is different. It fires at the **start of a turn**, when a prompt is submitted and before Claude works on it.

In this notebook you will:

1. See the prompt inside a hook, in `input_data["prompt"]`.
2. Inject context with `additionalContext`, so Claude knows things the prompt never said.
3. Make that injection conditional on what the prompt contains.

One distinction runs through the whole notebook. **The prompt stays exactly as typed.** The `additionalContext` field adds extra information *beside* the prompt. `PreToolUse` guards actions. `UserPromptSubmit` supplies knowledge.

## Cell 1: Install the Claude Agent SDK

This cell installs the `claude-agent-sdk` Python package, which is the only package this notebook needs. The package bundles the Claude Code CLI that actually runs the agent loop, so there is nothing else to install.

| Part | Meaning |
|---|---|
| `==0.2.163` | Pins one exact release so the notebook behaves the way it does on screen. |
| `-q` | Quiet mode. Keeps the install output short. |

If the package is already present in your current Colab session, pip checks the version and finishes quickly. Run the cell and wait for it to finish before moving on.

In [1]:
# We pin the Claude Agent SDK to a specific version so every example runs
# exactly as shown in the course. To use the latest version, run:
# pip install -U claude-agent-sdk
# Or substitute your preferred version below. Newer versions may behave
# differently from the videos.
!pip install claude-agent-sdk==0.2.163 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 368.7/368.7 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 5.2 MB/s eta 0:00:00


## Cell 2: Configure your API key

The SDK reads your Anthropic API key from the `ANTHROPIC_API_KEY` environment variable. In Colab, the safe place to store the key is **Colab Secrets**, so the key never appears in the notebook itself.

**Add the secret (one time):**

1. Click the **key icon** in the left sidebar of Colab.
2. Click **Add new secret**.
3. Name it exactly `ANTHROPIC_API_KEY`.
4. Paste your key as the value. You create the key in the Claude Console at https://platform.claude.com/ under **API Keys**.
5. Switch **Notebook access** on.

The cell then reads the secret with `userdata.get(...)` and writes it into `os.environ`, where the SDK looks for it.

> **Running locally instead of Colab?** Skip this cell and set the `ANTHROPIC_API_KEY` environment variable in your terminal before starting Python.

In [2]:
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

## Cell 3: Choose the model

`MODEL_NAME` holds the model that every agent run in this notebook uses. Each `ClaudeAgentOptions` below passes `model=MODEL_NAME`, so **changing this one variable updates the model across the entire notebook**.

This course uses `claude-haiku-4-5` as its default model. To see the models that are currently available, use the link in the cell's comment.

In [3]:
# See latest models at: https://platform.claude.com/docs/en/about-claude/models/overview
MODEL_NAME = "claude-haiku-4-5"

## Cell 4: Create the sample project

The demos need something for the agent to read. This cell creates a small folder, `/content/kitchen_demo/`, holding four invented text files.

Three of them are kitchen prep summaries with **similar names and the same shape**: `prep-0411.txt`, `prep-0412.txt` and `prep-0413.txt`. Each contains exactly one unique, checkable fact about a walnut batch. A fourth file holds an unrelated storeroom note. None of the files explains any abbreviation.

That design is deliberate. Later you will ask about "KP1", a team abbreviation that exists nowhere in these files. Because each prep file has a different fact, the agent's answer tells you which file it used, and you can check that against the files on disk instead of trusting how the answer reads.

All of this data is invented for the lesson.

In [4]:
from pathlib import Path

demo_dir = Path("/content/kitchen_demo")
demo_dir.mkdir(parents=True, exist_ok=True)

files = {
    "prep-0411.txt": "Kitchen prep summary 0411.\nWalnut batch 3 is on schedule.\n",
    "prep-0412.txt": "Kitchen prep summary 0412.\nWalnut batch 7 is delayed until Thursday.\n",
    "prep-0413.txt": "Kitchen prep summary 0413.\nWalnut batch 9 is complete.\n",
    "storeroom-notes.txt": "Storeroom note.\nThe cold shelf is restocked every Monday.\n",
}

for name, text in files.items():
    (demo_dir / name).write_text(text)
    print(f"Created {name}")

Created prep-0411.txt
Created prep-0412.txt
Created prep-0413.txt
Created storeroom-notes.txt


## Cell 5: Verify the files

Before any agent runs, confirm what is actually on disk. This cell prints the folder tree and then the contents of every file.

Look for the one unique fact in each prep file: **batch 3** is on schedule, **batch 7** is delayed until Thursday, and **batch 9** is complete. These facts are the ground truth for the rest of the notebook.

In [5]:
print(f"{demo_dir.name}/")
for path in sorted(demo_dir.iterdir()):
    print(f"  {path.name}")

print()
for path in sorted(demo_dir.iterdir()):
    print(f"--- {path.name} ---")
    print(path.read_text())

kitchen_demo/
  prep-0411.txt
  prep-0412.txt
  prep-0413.txt
  storeroom-notes.txt

--- prep-0411.txt ---
Kitchen prep summary 0411.
Walnut batch 3 is on schedule.

--- prep-0412.txt ---
Kitchen prep summary 0412.
Walnut batch 7 is delayed until Thursday.

--- prep-0413.txt ---
Kitchen prep summary 0413.
Walnut batch 9 is complete.

--- storeroom-notes.txt ---
Storeroom note.
The cold shelf is restocked every Monday.



## Cell 6: The baseline, no hook

Before adding a hook, let's see what the agent does without one. This cell asks a question that uses an abbreviation the agent has no way to know: **"What is the status of KP1? Answer in one sentence."**

**The mental model to keep in mind:** `query()` is not a function that returns a value. It returns an **async stream of messages** while the agent loop runs on its own. Intermediate tool use arrives as separate messages in that stream, and the final answer arrives on the `ResultMessage`. That is why the cell uses `async for` and checks each message for a `result` attribute.

| Option | Why it is set this way |
|---|---|
| `allowed_tools=["Read", "Glob"]` | Least privilege. The agent can look at files and nothing else. Read tools are fine here because this lecture does not demonstrate a permission gate. |
| `model=MODEL_NAME` | Uses the model from Cell 3. |
| `cwd="/content"` | Sets the agent's working folder. |

The cell also prints the name of each tool the agent calls, so you can see what it does on the way to an answer.

**What to look for:** the agent may search files, may guess, or may say it does not know what KP1 means. Real agents vary from run to run, so the exact wording will differ. What matters is that nothing told the agent what KP1 means.

In [6]:
from claude_agent_sdk import (
    query,
    ClaudeAgentOptions,
    AssistantMessage,
    ToolUseBlock,
)

async for message in query(
    prompt="What is the status of KP1? Answer in one sentence.",
    options=ClaudeAgentOptions(
        allowed_tools=["Read", "Glob"],
        model=MODEL_NAME,
        cwd="/content",
    ),
):
    if isinstance(message, AssistantMessage):
        for block in message.content:
            if isinstance(block, ToolUseBlock):
                print(f"[tool] {block.name}")
    if hasattr(message, "result"):
        print()
        print("Answer:", message.result)

[tool] Bash
[tool] Bash
[tool] Bash
[tool] Bash

Answer: I cannot find any reference to "KP1" in the current directory or its contents. Without more context about what KP1 refers to (whether it's a project identifier, task, key performance indicator, or something else), I'm unable to provide its status. Could you provide more details about what KP1 is?


## Cell 7: Inspect the prompt with a hook

Now the first hook. A `UserPromptSubmit` hook receives the submitted prompt before Claude works on it. This cell registers a hook that only **prints** the incoming prompt and returns `{}`, which changes nothing.

**How a hook is wired in:**

- The callback signature is `async def hook(input_data, tool_use_id, context)`. For this event, `tool_use_id` is not meaningful, and `context` is reserved for future use.
- `input_data["prompt"]` holds the text of the submitted prompt. The same dictionary also carries fields such as `session_id`, `cwd` and `hook_event_name`.
- Hooks are registered in `ClaudeAgentOptions(hooks={...})`. The key is the event name, `"UserPromptSubmit"`, and the value is a list of `HookMatcher` objects.
- We register with **no matcher**: `HookMatcher(hooks=[show_prompt])`. Matchers filter on tool names, and the hooks reference lists `UserPromptSubmit` as an event with no matcher support.

Two practical notes. First, `UserPromptSubmit` hooks have a default timeout of 30 seconds, and if a callback times out, Claude Code blocks the prompt, so keep these hooks fast. Second, the printed prompt is shown with `!r`, which wraps it in quotes so you can see its exact text.

**What to look for:** the `[hook]` line should appear **before** any `[tool]` line, because the hook runs before Claude starts working on the prompt. The prompt and the agent are the same as in Cell 6.

In [7]:
from claude_agent_sdk import HookMatcher


async def show_prompt(input_data, tool_use_id, context):
    print(f"[hook] Prompt received: {input_data['prompt']!r}")
    return {}


async for message in query(
    prompt="What is the status of KP1? Answer in one sentence.",
    options=ClaudeAgentOptions(
        allowed_tools=["Read", "Glob"],
        model=MODEL_NAME,
        cwd="/content",
        hooks={
            "UserPromptSubmit": [HookMatcher(hooks=[show_prompt])],
        },
    ),
):
    if isinstance(message, AssistantMessage):
        for block in message.content:
            if isinstance(block, ToolUseBlock):
                print(f"[tool] {block.name}")
    if hasattr(message, "result"):
        print()
        print("Answer:", message.result)

[hook] Prompt received: 'What is the status of KP1? Answer in one sentence.'
[tool] Bash
[tool] Grep
[tool] Bash
[tool] Read
[tool] Read
[tool] Read

Answer: I cannot find any reference to "KP1" in the current directory or its files. Could you provide more context about what KP1 refers to (e.g., a project file, task ID, key performance indicator, or other reference)?


## Cell 8: Inject context with `additionalContext`

This is the cell the lecture is about. The hook now returns extra information for Claude, and the prompt itself is untouched.

The return value has this shape:

```python
{
    "hookSpecificOutput": {
        "hookEventName": "UserPromptSubmit",
        "additionalContext": "...text for Claude...",
    }
}
```

| Field | Meaning |
|---|---|
| `hookSpecificOutput` | The container for event-specific output. |
| `hookEventName` | Must name the event the output belongs to, here `"UserPromptSubmit"`. |
| `additionalContext` | Text added to Claude's context alongside the submitted prompt. |

**The prompt stays exactly as typed.** The Python types offer no field for rewriting the prompt. The hook can only place extra information next to it.

**Design rule: keep injected context short, factual and relevant.** It is added on every turn the hook runs for, so a short factual statement works better than a long list of instructions. The hooks reference also notes that each `additionalContext` string is capped at 10,000 characters.

The glossary below is one sentence of fact: *KP1 means the kitchen prep summary file `prep-0412.txt`.*

**Ground truth.** The cell runs the same prompt as Cell 6 **twice**, then checks each final answer for the unique marker of each prep file: "batch 3", "batch 7" and "batch 9". The answer wording varies, but the batch number tells you which file the agent used. If the context reaches Claude, you should see **batch 7** found in both runs and the other two missing.

A note on safety: a `UserPromptSubmit` hook that starts subagents can trigger itself in a loop. This hook does not start anything, so it is safe.

In [8]:
GLOSSARY = (
    "Team glossary: KP1 means the kitchen prep summary file "
    "/content/kitchen_demo/prep-0412.txt."
)


async def add_glossary(input_data, tool_use_id, context):
    return {
        "hookSpecificOutput": {
            "hookEventName": "UserPromptSubmit",
            "additionalContext": GLOSSARY,
        }
    }


markers = {
    "prep-0411 (batch 3)": "batch 3",
    "prep-0412 (batch 7)": "batch 7",
    "prep-0413 (batch 9)": "batch 9",
}

for run in (1, 2):
    answer = ""
    async for message in query(
        prompt="What is the status of KP1? Answer in one sentence.",
        options=ClaudeAgentOptions(
            allowed_tools=["Read", "Glob"],
            model=MODEL_NAME,
            cwd="/content",
            hooks={
                "UserPromptSubmit": [HookMatcher(hooks=[add_glossary])],
            },
        ),
    ):
        if hasattr(message, "result"):
            answer = message.result

    print(f"Run {run} answer: {answer}")
    for label, marker in markers.items():
        found = marker in answer.lower()
        print(f"  {label}: {'FOUND' if found else 'not found'}")
    print()

Run 1 answer: KP1 shows that Walnut batch 7 is delayed until Thursday.
  prep-0411 (batch 3): not found
  prep-0412 (batch 7): FOUND
  prep-0413 (batch 9): not found

Run 2 answer: Walnut batch 7 is delayed until Thursday.
  prep-0411 (batch 3): not found
  prep-0412 (batch 7): FOUND
  prep-0413 (batch 9): not found



## Cell 9: Inject conditionally

Cell 8 injected the glossary on **every** prompt, even when the prompt had nothing to do with KP1. That is wasteful, because injected context is added on every turn. A better hook decides.

This hook checks `input_data["prompt"]` for the word "KP1", ignoring case, and injects the glossary **only then**. A module-level dictionary, `hook_stats`, counts two things: how many times the hook ran, and how many times it injected.

The cell then makes **two separate `query()` calls**, back to back:

1. A prompt that mentions KP1.
2. An unrelated prompt: *List the file names in /content/kitchen_demo.*

**What to look for in the counters:** the hook runs for both prompts, but it injects for only one. Printing the counters after each call lets you watch that happen. The hook runs on every prompt. The injection is a decision you code.

In [9]:
hook_stats = {"ran": 0, "injected": 0}


async def glossary_if_kp1(input_data, tool_use_id, context):
    hook_stats["ran"] += 1

    if "kp1" in input_data["prompt"].lower():
        hook_stats["injected"] += 1
        return {
            "hookSpecificOutput": {
                "hookEventName": "UserPromptSubmit",
                "additionalContext": GLOSSARY,
            }
        }

    return {}


prompts = [
    "What is the status of KP1? Answer in one sentence.",
    "List the file names in /content/kitchen_demo.",
]

for prompt in prompts:
    print(f"Prompt: {prompt}")
    async for message in query(
        prompt=prompt,
        options=ClaudeAgentOptions(
            allowed_tools=["Read", "Glob"],
            model=MODEL_NAME,
            cwd="/content",
            hooks={
                "UserPromptSubmit": [HookMatcher(hooks=[glossary_if_kp1])],
            },
        ),
    ):
        if hasattr(message, "result"):
            print("Answer:", message.result)
    print(f"Counters so far: {hook_stats}")
    print()

Prompt: What is the status of KP1? Answer in one sentence.
Answer: Walnut batch 7 is delayed until Thursday.
Counters so far: {'ran': 1, 'injected': 1}

Prompt: List the file names in /content/kitchen_demo.
Answer: The files in `/content/kitchen_demo` are:

1. prep-0411.txt
2. prep-0412.txt
3. prep-0413.txt
4. storeroom-notes.txt
Counters so far: {'ran': 2, 'injected': 1}



## Cell 10: Final recap

The last cell prints two things: the hook counters from Cell 9, and the contents of the sample folder.

The counters should show that the hook ran once for each of the two `query()` calls in Cell 9 and injected once. The folder listing should be unchanged, because nothing in this lecture wrote a file. Every behavior you saw came from context, not from actions.

**Key decision:** supply knowledge with `UserPromptSubmit`, and guard actions with `PreToolUse`.

In [10]:
print("Hook counters:", hook_stats)
print()

print(f"{demo_dir.name}/")
for path in sorted(demo_dir.iterdir()):
    print(f"  {path.name}")

Hook counters: {'ran': 2, 'injected': 1}

kitchen_demo/
  prep-0411.txt
  prep-0412.txt
  prep-0413.txt
  storeroom-notes.txt
